# Pratica — Lezione 5: Introduzione alle reti neurali convoluzionali

Curso **Deep Learning con Python y PyTorch desde cero** · DIH DATAlife

En esta práctica trabajas sobre arquitecturas de CNN (Convolutional Neural Network) sin entrenarlas todavía (eso llega en la clase 6). Hay dos ejercicios:

1. Añadir un segundo bloque convolucional a una CNN ya empezada y calcular la forma resultante
2. Comprobar, con un forward pass sobre un lote sintético, que la forma de salida coincide con el número de clases esperado

Completa las partes marcadas con `# TODO:` antes de mirar la solución. Ejecuta primero la celda de preparación, que es común a los dos ejercicios.

## Preparazione comune

Questa cella importa `torch` e i moduli necessari per definire strati convoluzionali e di pooling. Non serve caricare MNIST: in questa pratica lavori con lotti di immagini sintetiche (`torch.randn`) della stessa forma di un lotto reale di MNIST, per poter verificare l'architettura senza aspettare alcun download.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Versión de PyTorch:', torch.__version__)

## Esercizio 1 · Aggiungi un secondo blocco convoluzionale

**Enunciato:** la classe `CNNUnBloque` di seguito ha solo un blocco `nn.Conv2d` + `nn.MaxPool2d`. Il tuo compito è aggiungere un **secondo** blocco (`conv2` + `pool2`) antes de aplanar el tensor, ajustar la capa `nn.Linear` final al nuevo tamaño y comprobar con un forward pass que la salida tiene la forma esperada.

**Pistas:**

- Usa 32 filtros y un tamaño de filtro de 3x3 en `conv2`, con `stride=1` y `padding=1`, igual que en el notebook de teoría
- La imagen de entrada mide 28x28. Después del primer `pool1` (2x2) mide 14x14. Una convolución con `padding=1` y `kernel_size=3` no cambia el alto ni el ancho, así que sigue siendo 14x14 tras `conv2`. Después de `pool2` (2x2) queda en 7x7
- El número de entradas de `nn.Linear` es `canales_de_salida * alto * ancho` del último mapa de características, aplanado
- Verifica al final que `salida.shape` es `(4, 10)`, porque el lote de prueba tiene 4 imágenes y el problema tiene diez clases

In [ ]:
class CNNUnBloque(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3,
                                stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # TODO: añade aquí un segundo bloque:
        #   self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3,
        #                           stride=1, padding=1)
        #   self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # TODO: ajusta el número de entradas de nn.Linear una vez calcules (ver
        #       las pistas del enunciado) la forma del tensor tras el segundo
        #       bloque de pooling. Con un solo bloque, la forma tras pool1 es
        #       (lote, 16, 14, 14), así que de momento la capa usa ese tamaño.
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # TODO: aplica aquí conv2 + ReLU, seguido de pool2

        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


modelo_dos_bloques = CNNUnBloque()

# Lote sintético con la misma forma que un lote real de MNIST: 4 imágenes de 1 canal, 28x28
lote_ejemplo = torch.randn(4, 1, 28, 28)
salida = modelo_dos_bloques(lote_ejemplo)

print('Forma de la salida:', salida.shape)  # debería ser torch.Size([4, 10])

<details>
<summary>💡 Vedi soluzione</summary>

```python
class CNNUnBloque(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3,
                                stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Segundo bloque convolucional: 16 canales de entrada (los filtros del
        # primer bloque), 32 filtros nuevos de salida
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3,
                                stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Cálculo de la forma final, paso a paso:
        #   entrada:        28x28
        #   tras pool1:     14x14   (28 // 2)
        #   tras conv2:     14x14   (padding=1 y kernel_size=3 no cambian el tamaño)
        #   tras pool2:      7x7    (14 // 2)
        # La capa densa recibe 32 canales de 7x7 aplanados: 32 * 7 * 7 = 1568
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


modelo_dos_bloques = CNNUnBloque()

lote_ejemplo = torch.randn(4, 1, 28, 28)
salida = modelo_dos_bloques(lote_ejemplo)

print('Forma de la salida:', salida.shape)  # torch.Size([4, 10])
```
</details>

## Esercizio 2 · Controlla la forma di uscita con un batch sintetico

**Enunciato:** definisci una CNN completa (puoi riutilizzare l'architettura `CNNSimple` del notebook di teoria, con dos bloques de convolución + pooling y una capa `nn.Linear` final) y comprueba, con un `assert`, que un forward pass sobre un lote sintético de 16 imágenes produce una salida con forma `(16, 10)`.

**Pistas:**

- El lote sintético debe tener la misma forma que un lote real de MNIST: `(16, 1, 28, 28)`
- `NUM_CLASES` ya está definido como diez: úsalo en la capa `nn.Linear` final en lugar de escribir el número directamente
- Un `assert forma == (16, NUM_CLASES)` lanza un error si la forma no coincide, así que si tu código no falla al ejecutar la celda, la arquitectura está bien dimensionada
- Recuerda que `torch.Size` se puede comparar directamente con una tupla de Python, por ejemplo `salida.shape == (16, 10)`

In [ ]:
NUM_CLASES = 10


class CNNParaVerificar(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define conv1 (1 -> 16 canales, kernel_size=3, stride=1, padding=1)
        #       y pool1 (MaxPool2d de 2x2)

        # TODO: define conv2 (16 -> 32 canales, kernel_size=3, stride=1, padding=1)
        #       y pool2 (MaxPool2d de 2x2)

        # TODO: define fc como nn.Linear con el tamaño de entrada correcto
        #       (recuerda el cálculo del ejercicio 1) y NUM_CLASES de salida
        pass

    def forward(self, x):
        # TODO: implementa el forward pass completo: conv1 + ReLU, pool1,
        #       conv2 + ReLU, pool2, aplanar y fc
        pass


modelo_verificacion = CNNParaVerificar()

# Lote sintético de imágenes con la misma forma que un lote real de MNIST
lote_sintetico = torch.randn(16, 1, 28, 28)

# TODO: haz el forward pass sobre lote_sintetico y guarda el resultado
#       en la variable `salida`
salida = None

# TODO: descomenta la siguiente línea una vez tengas `salida` bien calculada
# assert salida.shape == (16, NUM_CLASES), f'Forma inesperada: {salida.shape}'
# print('Forma de la salida correcta:', salida.shape)

<details>
<summary>💡 Vedi soluzione</summary>

```python
NUM_CLASES = 10


class CNNParaVerificar(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3,
                                stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3,
                                stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = nn.Linear(32 * 7 * 7, NUM_CLASES)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


modelo_verificacion = CNNParaVerificar()

lote_sintetico = torch.randn(16, 1, 28, 28)

salida = modelo_verificacion(lote_sintetico)

assert salida.shape == (16, NUM_CLASES), f'Forma inesperada: {salida.shape}'
print('Forma de la salida correcta:', salida.shape)  # torch.Size([16, 10])
```
</details>

## Per continuare a esercitarti

Se vuoi andare un po' oltre:

- Ripeti l'esercizio 1 ma cambia `padding=1` con `padding=0` in `conv2`. Che forma ha il tensore dopo `conv2` ora? Ricalcola a mano la dimensione di input di `nn.Linear` e correggi il codice perché non fallisca
- Aggiungi un terzo blocco `nn.Conv2d` + `nn.MaxPool2d` all'architettura dell'esercizio 2 e calcola di nuovo la forma finale prima dello strato `nn.Linear`

Quando hai finito, torna al capitolo 5 del manuale e confronta i tuoi calcoli con il riepilogo dei punti chiave: le forme che hai ottenuto coincidono con quanto previsto dalla teoria? Nella lezione 6 userai un'architettura come questa per addestrare davvero su MNIST.